# PF4 CAD and MI Associations from CARDIoGRAMplusC4D GWAS

This notebook processes coronary artery disease and myocardial infarction summary statistics from the CARDIoGRAMplusC4D datasets to identify SNP associations within the PF4 genomic region. Variants are filtered based on genomic coordinates and cross-referenced with PF4 SNPs from NCBI dbSNP.

In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
region_path = Path("../data/interim/ensembl/region.json")
variants_path = Path("../data/interim/ncbi/variants.csv")

cad_path = Path("../data/raw/cardiogram_c4d/cad_harmonized.tsv")
mi_path = Path("../data/raw/cardiogram_c4d/mi_harmonized.tsv")

out_dir = Path("../data/interim/cardiogram_c4d")
out_dir.mkdir(parents=True, exist_ok=True)

out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    region = json.load(f)

chromosome = str(region["chromosome"])
region_start = int(region["region_start"])
region_end = int(region["region_end"])

chromosome, region_start, region_end

('4', 73930811, 74032027)

In [4]:
variants_df = pd.read_csv(variants_path)

variants_df.shape

(1976, 20)

In [5]:
pf4_variants_df = variants_df.drop_duplicates().copy()

pf4_variants_df["rsID"] = pf4_variants_df["rsID"].astype(str)

pf4_variants_df[["CHR_GRCh38", "POS_GRCh38"]] = (
    pf4_variants_df["CHRPOS_GRCh38"]
    .astype(str)
    .str.split(":", expand=True)
)

pf4_variants_df["CHR_GRCh38"] = pf4_variants_df["CHR_GRCh38"].astype(str)
pf4_variants_df["POS_GRCh38"] = pd.to_numeric(
    pf4_variants_df["POS_GRCh38"],
    errors="coerce"
).astype("Int64")

priority_pf4_rsids_set = set(
    pf4_variants_df.loc[
        pf4_variants_df["Priority"].isin(["Primary", "Secondary"]),
        "rsID"
    ]
)

pf4_variants_df[["rsID", "CHR_GRCh38", "POS_GRCh38", "Priority"]].head()

,rsID,CHR_GRCh38,POS_GRCh38,Priority
0,rs2233648,4,73981419,Primary
1,rs367951530,4,73981468,Below threshold
2,rs764474600,4,73981460,Below threshold
3,rs765704070,4,73981445,Below threshold
4,rs982761496,4,73981500,Below threshold


In [6]:
cardiogram_columns = [
    "hm_variant_id",
    "hm_rsid",
    "hm_chrom",
    "hm_pos",
    "hm_other_allele",
    "hm_effect_allele",
    "hm_beta",
    "hm_odds_ratio",
    "hm_ci_lower",
    "hm_ci_upper",
    "hm_effect_allele_frequency",
    "hm_code",
    "variant_id",
    "chromosome",
    "base_pair_location",
    "effect_allele",
    "other_allele",
    "effect_allele_frequency",
    "beta",
    "standard_error",
    "p_value",
    "odds_ratio",
    "ci_lower",
    "ci_upper",
]

In [7]:
def process_cardiogram_dataset(input_path, phenotype_label, column_prefix):
    df = pd.read_csv(
        input_path,
        sep="\t",
        usecols=lambda col: col in cardiogram_columns,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df["hm_chrom"] = df["hm_chrom"].astype(str)
    df["hm_pos"] = pd.to_numeric(df["hm_pos"], errors="coerce").astype("Int64")
    df["hm_rsid"] = df["hm_rsid"].astype(str)

    region_df = df[
        (df["hm_chrom"] == chromosome)
        & (df["hm_pos"] >= region_start)
        & (df["hm_pos"] <= region_end)
    ].copy()

    merged_df = region_df.merge(
        pf4_variants_df,
        left_on="hm_rsid",
        right_on="rsID",
        how="inner"
    )

    merged_df.insert(0, "phenotype", phenotype_label)

    rename_map = {
        col: f"{column_prefix}_{col}"
        for col in region_df.columns
    }

    merged_df = merged_df.rename(columns=rename_map)

    return merged_df

In [8]:
cad_annotated_df = process_cardiogram_dataset(
    input_path=cad_path,
    phenotype_label="Coronary artery disease",
    column_prefix="CAD"
)

cad_annotated_df.shape

(8, 47)

In [9]:
mi_annotated_df = process_cardiogram_dataset(
    input_path=mi_path,
    phenotype_label="Myocardial infarction",
    column_prefix="MI"
)

mi_annotated_df.shape

(0, 47)

In [10]:
cad_annotated_df.to_csv(out_cad_csv, index=False)
mi_annotated_df.to_csv(out_mi_csv, index=False)

print("Saved:", out_cad_csv)
print("Saved:", out_mi_csv)

Saved: ../data/interim/cardiogram_c4d/cad_associations.csv
Saved: ../data/interim/cardiogram_c4d/mi_associations.csv


In [11]:
cad_priority_matches = int(
    cad_annotated_df["Priority"].isin(["Primary", "Secondary"]).sum()
)

mi_priority_matches = int(
    mi_annotated_df["Priority"].isin(["Primary", "Secondary"]).sum()
)

summary = {
    "source_dataset": "CARDIoGRAMplusC4D harmonized GWAS summary statistics",
    "genome_assembly": "GRCh38",
    "total_pf4_variants": int(len(pf4_variants_df)),
    "priority_pf4_variants": int(len(priority_pf4_rsids_set)),
    "cad": {
        "phenotype": "Coronary artery disease",
        "pf4_matches": int(len(cad_annotated_df)),
        "priority_matches": cad_priority_matches,
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "pf4_matches": int(len(mi_annotated_df)),
        "priority_matches": mi_priority_matches,
    },
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_dataset': 'CARDIoGRAMplusC4D harmonized GWAS summary statistics',
 'genome_assembly': 'GRCh38',
 'total_pf4_variants': 1976,
 'priority_pf4_variants': 176,
 'cad': {'phenotype': 'Coronary artery disease',
  'pf4_matches': 8,
  'priority_matches': 8},
 'mi': {'phenotype': 'Myocardial infarction',
  'pf4_matches': 0,
  'priority_matches': 0}}